# SpeechLM API in vLLM example

In [ ]:
from transformers import AutoModelForCausalLM
import torch

# load token embeddings and lm head to use outside of vllm
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
hf_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)
embedding_layer = hf_model.get_input_embeddings().eval()
lm_head = hf_model.lm_head.eval()
del hf_model

In [ ]:
# example of llm streaming usage

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

engine_args = AsyncEngineArgs(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_model_len=256,
    gpu_memory_utilization=0.8,
    return_hidden_states=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=20, temperature=0.7, top_p=0.9)

noise = 3.0
inputs = {
    "prompt": "My name is",
    "custom_inputs": {
        "some_embs": torch.randn(4, 2048, dtype=torch.bfloat16) * noise,
    }
}
async for output in engine.generate(
    inputs,
    sampling_params=sampling_params,
    request_id="1",
):
    print(output.outputs[0].text, flush=True)
    if not output.finished:
        await engine.append_request(request_id="1", custom_inputs={"some_embs": torch.randn(1, 2048, dtype=torch.bfloat16) * noise})
